# Цифровые технологии в профессиональной деятельности
## Раздел 2. Деревья, сети, карты
## Семинар 7. Сетевой анализ

На прошлом занятии мы работали с географическими данными и строили интерактивные карты. Сегодня мы переходим к другому типу нетекстовых данных — **сетям и графам**.

**Материал:** генеалогический датасет династии Габсбургов (85 персон, только мужская линия).

**Чему научимся:**
- Понимать базовые понятия теории графов
- Строить графы с помощью `networkx`
- Вычислять сетевые метрики
- Делать интерактивные визуализации с помощью `pyvis`
- Превращать «сырые» исторические данные в структуру для графа (self-merge)

Рекомендуется запустить все ячейки сверху вниз — каждая часть опирается на предыдущую.

---
## Теоретический блок: граф как инструмент гуманитарного исследования

### Что такое граф?

**Граф (Network / Graph)** — абстрактный способ описать систему, состоящую из **объектов** и **отношений** между ними, это математическая структура. В зависимости от предметной области объектами могут быть персонажи пьесы, исторические деятели, города, научные статьи.

Формально граф — это пара $G = (V, E)$, где $V$ — множество вершин (vertices), $E$ — множество рёбер (edges). Ничего сложнее нам не понадобится.

### Узлы и рёбра

| Термин | Синонимы | Пример в нашем датасете |
|--------|----------|-------------------------|
| **Узел / Вершина** | Node, Vertex | Конкретный человек: Карл V, Максимилиан I |
| **Ребро / Связь** | Edge, Link | Отношение «является отцом» |

### Типология графов

**Неориентированный граф (Undirected Graph)** — связи симметричны. Если A связан с B, то и B связан с A. Пример: брак, совместное присутствие на сцене, факт знакомства. В библиотеке `networkx`: `nx.Graph()`.

![Неориентированный граф](https://upload.wikimedia.org/wikipedia/commons/thumb/b/bf/Undirected.svg/250px-Undirected.svg.png)

**Ориентированный граф (Directed Graph / DiGraph)** — связи имеют направление. Цитирование (А цитирует Б, но не наоборот), генеалогия (А — отец Б), вассальная зависимость. В библиотеке `networkx`: `nx.DiGraph()`. Наш датасет — ориентированный: стрелка идёт от отца к сыну.

![Ориентированный граф](https://upload.wikimedia.org/wikipedia/commons/thumb/a/a2/Directed.svg/330px-Directed.svg.png)

**Взвешенный граф (Weighted Graph)** — у каждого ребра есть числовое значение (вес). В сетевом анализе драмы весом может служить количество реплик между персонажами; в библиографии — число совместных публикаций.

![Взвешенный граф](https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/Network_flow_9_vertices_blank.svg/960px-Network_flow_9_vertices_blank.svg.png)

### Базовые метрики

**Степень узла (Degree)** — количество рёбер, подходящих к узлу. В ориентированном графе:
- **In-Degree** — входящие связи (сколько людей называют этого человека своим отцом, т.е. сколько у него сыновей в нашем датасете)
- **Out-Degree** — исходящие связи (есть ли у него отец в нашем датасете; для основателей линии — 0)

**Плотность сети (Density)** — отношение числа существующих рёбер к максимально возможному. Значение от 0 до 1: чем ближе к 1, тем теснее переплетена сеть.

$\text{Density} = \frac{|E|}{|V| \cdot (|V|-1)}$

Для ориентированного графа делитель $|V|(|V|-1)$, для неориентированного — $\frac{|V|(|V|-1)}{2}$.

### Метрики центральности (Centrality)

Центральность — это семейство мер важности узла. Разные метрики отвечают на разные вопросы:

**Degree Centrality** — нормированная степень узла: насколько узел популярен по прямым связям. Просто делим степень на $(n-1)$, чтобы получить значение от 0 до 1, независимо от размера сети.

**Betweenness Centrality (центральность по посредничеству)** — как часто данный узел оказывается на кратчайшем пути между всеми другими парами узлов. В истории так выявляют ключевых дипломатов, через которых шли все переговоры, или персонажей, связывающих разные социальные группы.

$$C_B(v) = \sum_{s \neq v \neq t} \frac{\sigma_{st}(v)}{\sigma_{st}}$$

где $\sigma_{st}$ — общее число кратчайших путей от $s$ до $t$, а $\sigma_{st}(v)$ — число тех из них, что проходят через $v$.

Для генеалогического дерева betweenness покажет стволовых предков, через которых проходят все родственные цепочки.

---
## Часть 0. Установка и импорт

Для работы нам понадобятся две новые библиотеки:
- `networkx` — математика графов: создание, анализ, метрики
- `pyvis` — интерактивная HTML-визуализация графов (аналог `folium`, только для сетей)

In [351]:
# Установка библиотек (нужно раскомментировать и выполнить один раз)
# !pip install networkx pyvis

In [352]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
from IPython.display import IFrame, display

---
## Часть 1. Знакомство с NetworkX на примере

Прежде чем работать с реальными данными, разберём логику `networkx` на маленьком рукотворном примере. Представим, что нам известны пять человек и несколько отношений между ними.

Сначала построим **неориентированный граф** — например, «кто с кем знаком».

In [353]:
# Создаём неориентированный граф
G = nx.Graph()

G

In [354]:
# Добавляем узлы вручную
G.add_node("Анна")
G.add_nodes_from(["Борис", "Вера", "Глеб", "Дина"])

In [355]:
# Добавляем рёбра — связи между узлами
G.add_edge("Анна", "Борис")
G.add_edge("Анна", "Вера")
G.add_edge("Анна", "Глеб")
G.add_edge("Борис", "Глеб")
G.add_edge("Глеб", "Дина")
G.add_edge("Вера", "Дина")

In [356]:
print("Узлы:", list(G.nodes()))
print("Рёбра:", list(G.edges()))
print("Число узлов:", G.number_of_nodes())
print("Число рёбер:", G.number_of_edges())

Узлы: ['Анна', 'Борис', 'Вера', 'Глеб', 'Дина']
Рёбра: [('Анна', 'Борис'), ('Анна', 'Вера'), ('Анна', 'Глеб'), ('Борис', 'Глеб'), ('Вера', 'Дина'), ('Глеб', 'Дина')]
Число узлов: 5
Число рёбер: 6


Теперь вычислим метрики. Степень узла — самая простая: просто выводим методом `.degree()`, сколько рёбер подходит к каждому узлу.

In [357]:
# Степень каждого узла
print("Степени узлов:")
for node, degree in G.degree():
    print(f"{node}: {degree}")

print()

Степени узлов:
Анна: 3
Борис: 2
Вера: 2
Глеб: 3
Дина: 2



In [358]:
# Плотность сети
density = nx.density(G)
print(f"Плотность сети: {density:.3f}")
print(f"{density*100:.1f}% от всех возможных связей существует")

Плотность сети: 0.600
60.0% от всех возможных связей существует


In [359]:
# Метрики центральности
degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G)

print(degree_centrality)

{'Анна': 0.75, 'Борис': 0.5, 'Вера': 0.5, 'Глеб': 0.75, 'Дина': 0.5}


In [360]:
print("Degree Centrality (нормированная степень):")
for node, val in sorted(degree_centrality.items(), reverse=True):
    print(f"  {node}: {val:.3f}")

print()
print("Betweenness Centrality (посредничество):")
for node, val in sorted(betweenness_centrality.items(), reverse=True):
    print(f"  {node}: {val:.3f}")

Degree Centrality (нормированная степень):
  Дина: 0.500
  Глеб: 0.750
  Вера: 0.500
  Борис: 0.500
  Анна: 0.750

Betweenness Centrality (посредничество):
  Дина: 0.083
  Глеб: 0.250
  Вера: 0.083
  Борис: 0.000
  Анна: 0.250


### Ориентированный граф (DiGraph)

Теперь построим **ориентированный граф** — связи имеют направление. Используем пример с генеалогией: стрелка идёт от отца к сыну.

Обратите внимание: единственное отличие в коде — `nx.DiGraph()` вместо `nx.Graph()`.

In [361]:
# Создаём ориентированный граф
DG = nx.DiGraph()

# Добавляем рёбра: (отец, сын) — стрелка от отца к сыну
DG.add_edge("Дед", "Отец")
DG.add_edge("Дед", "Дядя")
DG.add_edge("Дед", "Тётя")
DG.add_edge("Дядя", "Кузина")
DG.add_edge("Отец", "Я")
DG.add_edge("Отец", "Брат")
DG.add_edge("Отец", "Сестра")

print("Узлы:", list(DG.nodes()))
print("Рёбра:", list(DG.edges()))
print()

Узлы: ['Дед', 'Отец', 'Дядя', 'Тётя', 'Кузина', 'Я', 'Брат', 'Сестра']
Рёбра: [('Дед', 'Отец'), ('Дед', 'Дядя'), ('Дед', 'Тётя'), ('Отец', 'Я'), ('Отец', 'Брат'), ('Отец', 'Сестра'), ('Дядя', 'Кузина')]



In [362]:
# В ориентированном графе степень разделяется на in и out
print("In-degree (входящие — сколько сыновей у узла):")
for node, deg in DG.in_degree():
    print(f"  {node}: {deg}")

print()
print("Out-degree (исходящие — есть ли у узла отец в датасете):")
for node, deg in DG.out_degree():
    print(f"  {node}: {deg}")

In-degree (входящие — сколько сыновей у узла):
  Дед: 0
  Отец: 1
  Дядя: 1
  Тётя: 1
  Кузина: 1
  Я: 1
  Брат: 1
  Сестра: 1

Out-degree (исходящие — есть ли у узла отец в датасете):
  Дед: 3
  Отец: 3
  Дядя: 1
  Тётя: 0
  Кузина: 0
  Я: 0
  Брат: 0
  Сестра: 0


Метрики центральности для ориентированного графа можно считать методами неориентированных графов и специальными `in_` / `out_` методами. Результаты будут разными:

In [363]:
print(f"Метод для неориентированного графа: {nx.degree_centrality(DG)}")

print(f"Метод in_: {nx.in_degree_centrality(DG)}")

print(f"Метод out_: {nx.out_degree_centrality(DG)}")

Метод для неориентированного графа: {'Дед': 0.42857142857142855, 'Отец': 0.5714285714285714, 'Дядя': 0.2857142857142857, 'Тётя': 0.14285714285714285, 'Кузина': 0.14285714285714285, 'Я': 0.14285714285714285, 'Брат': 0.14285714285714285, 'Сестра': 0.14285714285714285}
Метод in_: {'Дед': 0.0, 'Отец': 0.14285714285714285, 'Дядя': 0.14285714285714285, 'Тётя': 0.14285714285714285, 'Кузина': 0.14285714285714285, 'Я': 0.14285714285714285, 'Брат': 0.14285714285714285, 'Сестра': 0.14285714285714285}
Метод out_: {'Дед': 0.42857142857142855, 'Отец': 0.42857142857142855, 'Дядя': 0.14285714285714285, 'Тётя': 0.0, 'Кузина': 0.0, 'Я': 0.0, 'Брат': 0.0, 'Сестра': 0.0}


### Атрибуты узлов и рёбер в NetworkX

Каждый узел и каждое ребро в NetworkX — это словарь. В него можно записать любую информацию: числа, строки, списки.

In [364]:
# Атрибуты можно задать сразу при добавлении узла или ребра
DG.add_node("Дед", surname="Васильев", birth_year=1941)
DG.add_edge("Дед", "Отец", relation="father_of", weight=1)

# Атрибуты хранятся как обычный словарь
print("Атрибуты узла:", DG.nodes["Дед"])
print("Атрибуты ребра:", DG.edges["Дед", "Отец"])

Атрибуты узла: {'surname': 'Васильев', 'birth_year': 1941}
Атрибуты ребра: {'relation': 'father_of', 'weight': 1}


In [365]:
# Атрибуты можно дописать в любой момент — как в обычный словарь
DG.nodes["Отец"]["birth_year"] = 1962
DG.nodes["Брат"]["birth_year"] = 1996

print(DG.nodes["Отец"])
print(DG.nodes["Брат"])

{'birth_year': 1962}
{'birth_year': 1996}


In [366]:
print(DG.adj) #AdjacencyView — специальный объект, который представляет структуру связей (ребер) графа в виде словаря словарей

{'Дед': {'Отец': {'relation': 'father_of', 'weight': 1}, 'Дядя': {}, 'Тётя': {}}, 'Отец': {'Я': {}, 'Брат': {}, 'Сестра': {}}, 'Дядя': {'Кузина': {}}, 'Тётя': {}, 'Кузина': {}, 'Я': {}, 'Брат': {}, 'Сестра': {}}


In [367]:
DG.nodes

NodeView(('Дед', 'Отец', 'Дядя', 'Тётя', 'Кузина', 'Я', 'Брат', 'Сестра'))

In [368]:
DG.out_edges

OutEdgeView([('Дед', 'Отец'), ('Дед', 'Дядя'), ('Дед', 'Тётя'), ('Отец', 'Я'), ('Отец', 'Брат'), ('Отец', 'Сестра'), ('Дядя', 'Кузина')])

In [369]:
DG.in_edges

InEdgeView([('Дед', 'Отец'), ('Дед', 'Дядя'), ('Дед', 'Тётя'), ('Дядя', 'Кузина'), ('Отец', 'Я'), ('Отец', 'Брат'), ('Отец', 'Сестра')])

In [370]:
# data=True — просим вернуть не только имена узлов, но и их атрибуты
for node, attrs in DG.nodes(data=True):
    print(f"{node}: {attrs}")

Дед: {'surname': 'Васильев', 'birth_year': 1941}
Отец: {'birth_year': 1962}
Дядя: {}
Тётя: {}
Кузина: {}
Я: {}
Брат: {'birth_year': 1996}
Сестра: {}


In [371]:
# Для рёбер аналогично: три значения при data=True — откуда, куда, атрибуты
for source, target, attrs in DG.edges(data=True):
    print(f"{source} -> {target}: {attrs}")

Дед -> Отец: {'relation': 'father_of', 'weight': 1}
Дед -> Дядя: {}
Дед -> Тётя: {}
Отец -> Я: {}
Отец -> Брат: {}
Отец -> Сестра: {}
Дядя -> Кузина: {}


Запись метрик в атрибуты узлов — стандартный паттерн работы с NetworkX. Сначала вычисляем метрику (получаем словарь `{узел: значение}`), затем записываем результат обратно в граф. Тогда вся информация об узле хранится в одном месте, и `PyVis` сможет её прочитать при визуализации.

In [372]:
# Вычисляем метрику — получаем словарь
centrality = nx.in_degree_centrality(DG)
print("Словарь метрик:", centrality)

# Записываем каждое значение как атрибут узла
for node in DG.nodes():
    DG.nodes[node]["centrality"] = centrality[node]

# Проверяем: теперь centrality хранится прямо внутри графа
for node, attrs in DG.nodes(data=True):
    print(f"{node}: {attrs}")

Словарь метрик: {'Дед': 0.0, 'Отец': 0.14285714285714285, 'Дядя': 0.14285714285714285, 'Тётя': 0.14285714285714285, 'Кузина': 0.14285714285714285, 'Я': 0.14285714285714285, 'Брат': 0.14285714285714285, 'Сестра': 0.14285714285714285}
Дед: {'surname': 'Васильев', 'birth_year': 1941, 'centrality': 0.0}
Отец: {'birth_year': 1962, 'centrality': 0.14285714285714285}
Дядя: {'centrality': 0.14285714285714285}
Тётя: {'centrality': 0.14285714285714285}
Кузина: {'centrality': 0.14285714285714285}
Я: {'centrality': 0.14285714285714285}
Брат: {'birth_year': 1996, 'centrality': 0.14285714285714285}
Сестра: {'centrality': 0.14285714285714285}


---
## Часть 2. Знакомство с PyVis: интерактивная визуализация

`pyvis` работает по той же логике, что и `folium`: мы создаём объект, передаём в него данные, и на выходе получаем HTML-файл с интерактивной визуализацией.

Ключевое удобство: `pyvis` умеет напрямую импортировать граф из `networkx` — не нужно пересобирать всё заново.

In [373]:
# Создаём объект Network
# bgcolor — цвет фона, font_color — цвет подписей
net = Network(
    height="600px",         # высота холста в пикселях
    width="100%",           # ширина — 100% от ширины ячейки в тетрадке
    bgcolor="#1a1a2e",    # цвет фона в формате HEX (тёмно-синий)
    font_color="white",     # цвет подписей у узлов
    directed=True           # граф ориентированный: рёбра будут отображаться со стрелками
)

# Импортируем граф из networkx
net.from_nx(DG)

# Включаем физику — узлы будут отталкиваться и находить равновесие
net.toggle_physics(True)

# Сохраняем как HTML и отображаем прямо в тетрадке
net.write_html("toy_graph.html")

**Что такое «физика»?** 

`pyvis` использует физический симулятор: узлы ведут себя как заряженные частицы, а рёбра — как пружины. При запуске они разлетаются от центра и постепенно находят устойчивое положение, где силы уравновешены, что позволяет автоматически получить читаемую раскладку без ручного задания координат.

### Параметры физического симулятора

PyVis использует библиотеку vis.js, и по умолчанию применяет алгоритм **Barnes-Hut** — физическую модель, где узлы отталкиваются друг от друга как заряженные частицы, а рёбра притягивают соединённые узлы как пружины.

Самые важные параметры:

**`gravitationalConstant`** (по умолчанию: -2000)

Сила отталкивания между всеми узлами. Чем больше отрицательное число — тем сильнее узлы разлетаются от центра. При значении близком к 0 узлы слипаются в кучу.

**Когда менять:** если узлы слишком плотно сгрудились — увеличьте отрицательное значение.

**`springLength`** (по умолчанию: 95) и **`springConstant`** (по умолчанию: 0.04)

`springLength` — желаемая длина ребра в пикселях: на каком расстоянии симулятор будет держать два связанных узла.

`springConstant` — жёсткость пружины: насколько настойчиво ребро тянет узлы к этому расстоянию. Высокое значение — граф собирается туго, низкое — узлы гуляют свободнее.

**Когда менять:** если связанные узлы расположены слишком далеко или слишком близко друг к другу.

**`damping`** (по умолчанию: 0.09)

Затухание — насколько быстро узлы успокаиваются и перестают двигаться. Значение от 0 до 1: при 0 узлы колышутся бесконечно, при 1 — замирают мгновенно.

**Когда менять:** если анимация слишком долго не останавливается — увеличьте значение.

Значения параметров по умолчанию работают для большинства небольших графов. Панель `show_buttons(filter_=['physics'])` нужна именно для того, чтобы подобрать нужные значения вручную в браузере, а потом зафиксировать их через `set_options()` в коде.

In [374]:
# Параметры физики можно задать прямо в коде, не используя панель в браузере
# Используйте, когда нужно зафиксировать конкретный вид графа для публикации

net.set_options("""
{
  "physics": {
    "barnesHut": {
      "gravitationalConstant": -1000,
      "springLength": 100,
      "springConstant": 0.02,
      "damping": 0.5
    }
  }
}
""")

net.write_html("toy_graph.html")

---
## Часть 3. Кастомизация: связываем метрики с визуализацией

Сделаем так, чтобы размер узла соответствовал его центральности. Чем важнее предок — тем крупнее его кружок на графе.

Для этого нужно передать метрики в атрибуты узлов графа `networkx` **до** того, как мы передадим его в `pyvis`.

### Атрибуты узлов в PyVis

Узлам графа можно задавать визуальные свойства через атрибуты.
PyVis автоматически читает их при импорте из networkx.

Основные атрибуты:
- `size` — размер кружка на холсте (число, обычно от 5 до 50)
- `title` — текст всплывающей подсказки при наведении курсора
- `color` — цвет узла (HEX-строка или название цвета)
- `label` — подпись, которая видна прямо на холсте (по умолчанию — имя узла)

In [375]:
# Начнём с самого простого: зададим одинаковый цвет всем узлам
for node in DG.nodes():
    DG.nodes[node]["color"] = "#e8a838"

# Проверяем: атрибуты хранятся прямо в объекте графа как словарь
print(DG.nodes["Отец"])

{'birth_year': 1962, 'centrality': 0.14285714285714285, 'size': 10, 'color': '#e8a838'}


In [376]:
# size задаём не константой, а формулой: базовый размер + надбавка за центральность
# без базового значения узлы с нулевой метрикой стали бы невидимы
for node in DG.nodes():
    DG.nodes[node]["size"] = 15 + centrality[node] * 60

# Смотрим, что получилось у конкретного узла
print("Отец:", DG.nodes["Отец"]["size"])
print("Дед:", DG.nodes["Дед"]["size"])

Отец: 23.57142857142857
Дед: 15.0


In [377]:
# Вычисляем дополнительную метрику betweenness на нашем игрушечном DiGraph

betweenness = nx.betweenness_centrality(DG)

In [378]:
# title — это HTML-строка, которая появится при наведении курсора
# \n работает как перенос строки внутри подсказки
for node in DG.nodes():
    DG.nodes[node]["title"] = (
        f"{node}\n"
        f"In-degree centrality: {centrality[node]:.3f}\n"
        f"Betweenness: {betweenness[node]:.3f}"
    )

print(DG.nodes["Отец"]["title"])

Отец
In-degree centrality: 0.143
Betweenness: 0.071


In [379]:
# Создаём новую сеть с обновлёнными атрибутами
net2 = Network(
    height="800px",
    width="100%",
    bgcolor="#a1a1c2",
    font_color="white",
    directed=True
)
net2.from_nx(DG)

# Добавляем интерактивное меню для настройки физики прямо в браузере
net2.show_buttons(filter_=["physics"])

net2.write_html("toy_graph_styled.html")

Наведите курсор на любой узел — должна появиться всплывающая подсказка со значениями метрик. Воспользуйтесь панелью «Physics» в правой части, чтобы изменить поведение симулятора без перезапуска кода.

---
## Часть 4. От сырых данных к графу

В реальных исторических данных информация о связях часто хранится неявно. Наш датасет — типичный пример: в нём есть `person_id` каждого человека и `father_id` — идентификатор его отца, но нет готовой таблицы `Source -> Target`.

### Загрузка и изучение данных

In [380]:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "enzomarx/habsburg-dynasty-male-line-genealogy-dataset",
  "FULL_HABSBURG_DYNASTY.csv",
)

C:\Users\NYX\AppData\Local\Temp\ipykernel_8852\515322102.py:5: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


In [381]:
print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов")
df.head(8)

Размер датасета: 84 строк, 9 столбцов


,person_id,name,birth_year,death_year,father_id,house,title,primary_territory,notes
0,E001,Eticho I Adalric,635.0,690,NaN,Etichonider,Count of Alsace,Alsace,Founder of Etichonider line
1,E002,Adalbert I,665.0,720,E001,Etichonider,Count of Alsace,Alsace,NaN
2,E003,Eticho II,700.0,723,E002,Etichonider,Count of Nordgau,Nordgau,NaN
3,E004,Alberic I,NaN,747,E003,Etichonider,Count of Nordgau,Nordgau,NaN
4,E005,Eberhard II,NaN,777,E004,Etichonider,Count of Nordgau,Nordgau,NaN
5,E006,Eberhard III of Dillingen,NaN,874,E005,Etichonider,Count of Nordgau,Nordgau,NaN
6,E007,Hugo III,NaN,940,E006,Etichonider,Count of Nordgau,Nordgau,NaN
7,E008,Guntram the Rich,920.0,973,E007,Etichonider,Count in Breisgau,Breisgau,Direct ancestor of Habsburgs


In [382]:
founders = df[df["father_id"].isna()]
print(f"Основатель династии: {len(founders)}\n")
print(founders[["name", "house", "notes"]])

Основатель династии: 1

               name        house                        notes
0  Eticho I Adalric  Etichonider  Founder of Etichonider line


In [383]:
# Распределение по ветвям
print("Распределение по ветвям (house):")
print(df["house"].value_counts())

print()
print("Распределение по титулам (топ-10):")
print(df["title"].value_counts().head(10))

Распределение по ветвям (house):
house
Habsburg       75
Etichonider     9
Name: count, dtype: int64

Распределение по титулам (топ-10):
title
Archduke of Austria      16
Holy Roman Emperor       12
co-Duke of Austria        7
Count of Habsburg         5
Count of Nordgau          5
Duke of Austria           5
Noble                     5
King of Spain             4
Duke of Lower Austria     3
King of Germany           2
Name: count, dtype: int64


### Проблема: одинаковые имена некоторых членов династии

In [384]:
df['name'].value_counts().head(10)

name
Rudolf II                             3
Albert III                            2
Albert IV                             2
Charles of Austria                    2
Eticho I Adalric                      1
Adalbert I                            1
Eticho II                             1
Guntram the Rich                      1
Eberhard IV                           1
Lanzelin of Klettgau and Altenburg    1
Name: count, dtype: int64

Давайте разграничим тёзок, добавив к их именам даты жизни.

In [385]:
# Находим имена, которые встречаются больше одного раза
duplicate_names = df[df.duplicated(subset='name', keep=False)]['name'].unique()
print("Имена с повторами:", duplicate_names)

Имена с повторами: ['Albert III' 'Rudolf II' 'Albert IV' 'Charles of Austria']


In [386]:
df['birth_year'] = df['birth_year'].astype('Int64')
df['death_year'] = df['death_year'].astype('Int64')

df[['birth_year', 'death_year']].fillna(0).head(5)

,birth_year,death_year
0,635,690
1,665,720
2,700,723
3,0,747
4,0,777


In [387]:
df['name_unique'] = df.apply(   # создаём новый столбец name_unique; apply применяет функцию к каждой строке

    lambda row: f"{row['name']} ({row['birth_year']} — {row['death_year']})" 
    if row['name'] in duplicate_names else row['name'],
    axis=1                                                      # axis=1 — применяем функцию по строкам (axis=0 — по столбцам)
)

print(df[df['name'].isin(duplicate_names)][['person_id', 'name', 'name_unique']])

   person_id                name                       name_unique
16      H008          Albert III          Albert III (<NA> — 1199)
17      H009           Rudolf II           Rudolf II (<NA> — 1232)
18      H010           Albert IV           Albert IV (1188 — 1239)
24      H016           Rudolf II           Rudolf II (1270 — 1290)
36      H028          Albert III          Albert III (1349 — 1395)
40      H032           Albert IV           Albert IV (1377 — 1404)
60      H052           Rudolf II           Rudolf II (1552 — 1612)
69      H061  Charles of Austria  Charles of Austria (1590 — 1624)
71      H063  Charles of Austria  Charles of Austria (1607 — 1632)


### Проблема: у нас есть узлы, но нет рёбер

Для создания графа нам нужна таблица вида:

| source (отец) | target (сын) |
|--------------|-------------|
| Eticho I Adalric | Adalbert I |
| Adalbert I | Eticho II |
| ... | ... |

В нашем датасете связь закодирована иначе: у каждого человека указан `father_id`. Нам нужно развернуть эту информацию.

### Слияние таблиц: соединяем таблицу саму с собой

Таблицу соединяем со своей копией: в левой части — потомки с их `father_id`. В правой — все персоны с их `person_id`. Совмещаем `father_id` из левой с `person_id` из правой — и получаем пары (отец, сын) по именам.

In [388]:
# Шаг 1. Посмотрим на левую и правую части до слияния
# Левая: потомки — те, у кого father_id заполнен
left = df[["name_unique", "father_id"]].dropna(subset=["father_id"])

# Правая: все персоны — служит справочником person_id -> name
right = df[["person_id", "name_unique"]].rename(columns={"name_unique": "father_name"})

In [389]:
print("Левая часть (сыновья):")
print(left.head(5).to_string(index=False))
print()

print("Правая часть (справочник отцов):")
print(right.head(5).to_string(index=False))

Левая часть (сыновья):
              name_unique father_id
               Adalbert I      E001
                Eticho II      E002
                Alberic I      E003
              Eberhard II      E004
Eberhard III of Dillingen      E005

Правая часть (справочник отцов):
person_id      father_name
     E001 Eticho I Adalric
     E002       Adalbert I
     E003        Eticho II
     E004        Alberic I
     E005      Eberhard II


In [390]:
# Шаг 2. Сопоставляем father_id из левой с person_id из правой
merged = pd.merge(
    left,
    right,
    left_on="father_id",   # из левой таблицы берём father_id
    right_on="person_id",  # сопоставляем с person_id из правой
    how="inner"            # оставляем только совпадения
)

### Типы объединения таблиц в pd.merge()

Параметр `how` определяет, какие строки попадут в результат,
если в одной из таблиц нет соответствующей записи.

| how | Что остаётся |
|-----|-------------|
| `inner` | только строки, у которых есть пара в обеих таблицах |
| `left` | все строки из левой таблицы; где пары нет — NaN |
| `right` | все строки из правой таблицы; где пары нет — NaN |
| `outer` | все строки из обеих таблиц; где пары нет — NaN |

В нашем случае выбираем `inner`, потому что нас интересуют только те пары «отец — сын», для которых оба человека есть в датасете.

Если бы мы использовали `left` — в результате остались бы сыновья, чьи отцы не вошли в датасет: в колонке `father_name` у них было бы `NaN`, и такое ребро нельзя добавить в граф. На самом деле такое ребро только одно, с основателем династии.

In [391]:
# Шаг 3. Формируем итоговую таблицу рёбер
edges = merged[["father_name", "name_unique"]].rename(
    columns={"father_name": "source", "name_unique": "target"}
)

print(f"Итоговая таблица рёбер: {len(edges)} строки")
print()
edges.head(10)

Итоговая таблица рёбер: 83 строки



,source,target
0,Eticho I Adalric,Adalbert I
1,Adalbert I,Eticho II
2,Eticho II,Alberic I
3,Alberic I,Eberhard II
4,Eberhard II,Eberhard III of Dillingen
5,Eberhard III of Dillingen,Hugo III
6,Hugo III,Guntram the Rich
7,Guntram the Rich,Eberhard IV
8,Guntram the Rich,Lanzelin of Klettgau and Altenburg
9,Lanzelin of Klettgau and Altenburg,Werner I


---
## Часть 5. Строим граф Габсбургов

Теперь у нас есть всё необходимое: таблица узлов и таблица рёбер, собираем граф.

In [392]:
# Создаём ориентированный граф
H = nx.DiGraph()

In [393]:
df.columns

Index(['person_id', 'name', 'birth_year', 'death_year', 'father_id', 'house',
       'title', 'primary_territory', 'notes', 'name_unique'],
      dtype='object')

In [394]:
# Добавляем узлы с атрибутами из датасета
for _, row in df.iterrows():
    H.add_node(
        row["name_unique"],
        house=row["house"],
        title=row["title"],
        birth_year=row["birth_year"] if pd.notna(row["birth_year"]) else "?",
        death_year=row["death_year"] if pd.notna(row["death_year"]) else "?",
        territory=row["primary_territory"]
    )

In [395]:
# Добавляем рёбра из таблицы, которую мы построили через self-merge
for _, row in edges.iterrows():
    H.add_edge(row["source"], row["target"])

In [396]:
print(f"Граф построен.")
print(f"Узлов: {H.number_of_nodes()}")
print(f"Рёбер: {H.number_of_edges()}")
print(f"Плотность: {nx.density(H):.4f}")

Граф построен.
Узлов: 84
Рёбер: 83
Плотность: 0.0119


### Анализ метрик

Вычислим метрики центральности и посмотрим, кто из Габсбургов оказывается наиболее «важным» с точки зрения структуры сети.

In [397]:
# In-degree: у кого больше всего сыновей в датасете
in_deg = nx.in_degree_centrality(H)

# Betweenness: кто чаще оказывается на «пути» между другими членами династии
betw = nx.betweenness_centrality(H)

# Собираем результаты в таблицу
metrics_df = pd.DataFrame({
    "name": list(H.nodes()),
    "in_degree_centrality": [in_deg[node] for node in H.nodes()],
    "betweenness_centrality": [betw[node] for node in H.nodes()]
})

In [398]:
metrics_df

,name,in_degree_centrality,betweenness_centrality
0,Eticho I Adalric,0.000000,0.000000
1,Adalbert I,0.012048,0.012048
2,Eticho II,0.012048,0.023803
3,Alberic I,0.012048,0.035263
4,Eberhard II,0.012048,0.046430
...,...,...,...
79,Ferdinand IV,0.012048,0.000000
80,Leopold I,0.012048,0.013224
81,Charles Joseph of Austria,0.012048,0.000000
82,Joseph I,0.012048,0.000000


In [399]:
print("Топ-10 по In-Degree Centrality (больше всего сыновей):")
metrics_df.sort_values("in_degree_centrality", ascending=False).head(10)[["name", "in_degree_centrality"]]


Топ-10 по In-Degree Centrality (больше всего сыновей):


,name,in_degree_centrality
1,Adalbert I,0.012048
2,Eticho II,0.012048
3,Alberic I,0.012048
4,Eberhard II,0.012048
12,Werner I the Pious,0.012048
5,Eberhard III of Dillingen,0.012048
6,Hugo III,0.012048
7,Guntram the Rich,0.012048
8,Eberhard IV,0.012048
9,Lanzelin of Klettgau and Altenburg,0.012048


In [400]:
print("Топ-10 по Betweenness Centrality (ключевые 'связующие' предки):")
metrics_df.sort_values("betweenness_centrality", ascending=False).head(10)[["name", "betweenness_centrality"]]

Топ-10 по Betweenness Centrality (ключевые 'связующие' предки):


,name,betweenness_centrality
29,Frederick I III the Fair,0.140170
28,Rudolf I III of Bohemia,0.137379
18,Albert IV (1188 — 1239),0.136644
17,Rudolf II (<NA> — 1232),0.135763
31,Albert II the Wise,0.133999
26,Rudolf I of Germany,0.133999
16,Albert III (<NA> — 1199),0.127975
14,Werner II,0.121657
45,Albert II of Germany,0.119600
40,Albert IV (1377 — 1404),0.117249


**Как интерпретировать результат?**

Высокое **in-degree** у правителя означает, что у него много сыновей в нашем датасете — он дал много ветвей. 

Высокое **betweenness** означает нечто другое: этот человек выступает мостом между разными ветвями. Если убрать его из графа, часть династического дерева окажется изолирована от другой части.

Совпадают ли лидеры по двум метрикам?

---
## Часть 6. Визуализация

Теперь соберём всё вместе: передадим метрики в визуальные атрибуты и построим интерактивный граф всей династии.

In [401]:
# Цвета для разных ветвей
house_colors = {
    "Etichonider": "#e8a838",  # золотой — ранняя ветвь
    "Habsburg":    "#4a90d9"   # синий — основная ветвь
}

# Добавляем метрики и атрибуты визуализации в узлы графа
for node in H.nodes():
    H.nodes[node]["size"] = 10 + betw[node] * 150
    H.nodes[node]["color"] = house_colors.get(H.nodes[node].get("house", ""), "#aaaaaa")
    H.nodes[node]["title"] = (
        f"<b>{node}</b><br>"
        f"Ветвь: {H.nodes[node].get('house', '?')}<br>"
        f"Титул: {H.nodes[node].get('title', '?')}<br>"
        f"Жизнь: {H.nodes[node].get('birth_year', '?')}–{H.nodes[node].get('death_year', '?')}<br>"
        f"Территория: {H.nodes[node].get('territory', '?')}<br>"
        f"Betweenness: {betw[node]:.4f}"
    )

print("Атрибуты обновлены.")

Атрибуты обновлены.


In [402]:
# Строим финальный граф
net_hab = Network(
    height="900px",
    width="100%",
    bgcolor="#0f0f23",
    font_color="#eeeeee",
    directed=True
)

net_hab.from_nx(H)
net_hab.show_buttons(filter_=["physics"])

net_hab.write_html("habsburg_network.html")

**Как работать с графом:**
- Наведите курсор на узел — увидите всплывающую карточку с данными о персоне
- Перетаскивайте узлы мышью, чтобы перестроить раскладку
- Используйте панель «Physics» для изменения параметров симулятора
- Крупные узлы — ключевые «связующие» предки (высокий betweenness)
- Жёлтые — Etichonider (предшественники), синие — Habsburgs

---
## Обсуждение: ограничения модели

Любая сетевая модель — это **упрощение**. Прежде чем делать исторические выводы, важно понять, что мы потеряли при формализации:

1. **Только мужская линия.** В нашем датасете нет женщин. Но именно через браки Габсбурги строили свои политические союзы (`bella gerant alii, tu felix Austria nube` — «пусть другие воюют, ты же, счастливая Австрия, заключай браки»). Без матримониальных рёбер граф неполон.

2. **Связь «отец–сын» — не единственная.** Политические союзы, опека, усыновление, брак — всё это тоже связи. Граф с одним типом рёбер не отражает всей сложности.

Сетевые метрики нужно интерпретировать с учётом структуры источника.

---
## Итоги

Сегодня мы:
1. Разобрали базовые понятия теории графов: узлы, рёбра, ориентированность, метрики
2. Познакомились с `networkx`: создали граф вручную, вычислили степени и центральности
3. Познакомились с `pyvis`: превратили математический граф в интерактивную HTML-страницу
4. Научились передавать результаты вычислений в визуальные атрибуты (размер, цвет, подсказки)
5. Разобрали механику **self-merge** в pandas: как извлечь таблицу рёбер из данных с `father_id`
6. Построили и проанализировали граф династии Габсбургов

### Документация
- NetworkX: https://networkx.org/documentation/stable/
- PyVis: https://pyvis.readthedocs.io/
- Pandas merge: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html

### Для дальнейшего изучения
- **Gephi** — десктопный инструмент для визуализации больших сетей (десятки тысяч узлов)
- **[Programming Historian: From Hermeneutics to Data to Networks](https://programminghistorian.org/en/lessons/creating-network-diagrams-from-historical-sources)** — пошаговый туториал по сетевому анализу исторических источников
- **Palladio (Stanford)** — браузерный инструмент для гуманитарных сетей без программирования